In [ ]:
import torch
import os
from pathlib import Path
from PIL import Image
import numpy as np
from transformers import pipeline
from dotenv import load_dotenv
import requests
from diffusers import (
    ControlNetModel,
    StableDiffusionControlNetPipeline,
    UniPCMultistepScheduler,
)

def download_image(url, save_path):
    """Download image from URL"""
    response = requests.get(url, stream=True)
    if response.status_code == 200:
        with open(save_path, 'wb') as f:
            f.write(response.content)
        return True
    return False

def main():
    # Create necessary directories
    Path("./tests/images").mkdir(parents=True, exist_ok=True)

    # Download sample image
    image_urls = [
        "https://huggingface.co/datasets/hf-internal-testing/diffusers-images/resolve/main/controlnet/room.png",
        "https://raw.githubusercontent.com/CompVis/stable-diffusion/main/assets/stable-samples/img2img/sketch-mountains-input.jpg"
    ]

    input_path = "tests/images/input.jpg"

    # Try downloading from each URL until successful
    for url in image_urls:
        if download_image(url, input_path):
            break
        else:
            print(f"Failed to download from {url}, trying next...")

    # Load environment variables
    load_dotenv()
    
    if not hf_token:
        raise ValueError("HF_TOKEN not found in environment variables")

    # Model checkpoint
    checkpoint = "lllyasviel/sd-controlnet-depth"

    # Load and process input image
    try:
        image = Image.open(input_path)
        if image.mode != "RGB":
            image = image.convert("RGB")
    except Exception as e:
        print(f"Error loading image from {input_path}: {e}")
        raise

    # Set your prompt
    prompt = "A cozy living room with modern furniture and warm lighting"

    # Initialize depth estimator and generate depth map
    print("Initializing depth estimator...")
    depth_estimator = pipeline('depth-estimation', use_auth_token=hf_token)
    depth = depth_estimator(image)['depth']
    depth = np.array(depth)
    depth = depth[:, :, None]
    depth = np.concatenate([depth, depth, depth], axis=2)
    control_image = Image.fromarray(depth)

    # Save depth map
    control_image.save("./tests/images/depth_map.png")
    print("Depth map saved to ./tests/images/depth_map.png")

    # Initialize models
    print("Loading ControlNet model...")
    controlnet = ControlNetModel.from_pretrained(
        checkpoint, 
        torch_dtype=torch.float16,
        use_auth_token=hf_token
    )

    print("Loading Stable Diffusion pipeline...")
    pipe = StableDiffusionControlNetPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5", 
        controlnet=controlnet, 
        torch_dtype=torch.float16,
        use_auth_token=hf_token,
        safety_checker=None
    )

    # Configure pipeline
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
    pipe.enable_model_cpu_offload()

    # Generate image
    print("Generating image...")
    generator = torch.manual_seed(0)
    output_image = pipe(
        prompt, 
        num_inference_steps=30, 
        generator=generator, 
        image=control_image
    ).images[0]

    # Save output
    output_image.save('./generated_output.png')
    print("Generated image saved to ./tests/images/generated_output.png")

if __name__ == "__main__":
    main()